# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [54]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)
import pandas as pd, numpy as np, scipy.stats as ss, statsmodels.api as sm

df = pd.read_csv('./customers.csv')

df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [55]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column
def missing_values():
    print("Fill Missing Value")
    null_value = df.isna().sum()
    print(null_value.to_string())

    print("\nRegion Labels")
    print(f"Unique values: {(df['region'].unique())}")

    print("\nImpossible Values")
    print(f"Values: {(df['income'].describe())}")
missing_values()



Fill Missing Value
customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0

Region Labels
Unique values: <StringArray>
['West', 'South', 'East', 'east']
Length: 4, dtype: str

Impossible Values
Values: count     38.000000
mean      83.007895
std       27.628563
min      -50.000000
25%       72.975000
50%       84.700000
75%       98.500000
max      114.500000
Name: income, dtype: float64


In [56]:
median_val = df['income'].median()
df['income'] = df['income'].fillna(median_val)

df['region'] = df['region'].replace(["east"], "East", regex=True)

df.loc[df['income'] == -50.0, 'income'] = np.nan

df['income'] = df['income'].fillna(df['income'].median())
df.head(40)

,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,84.7,49.9,South,No
4,5,43,72.4,48.3,East,No
5,6,44,84.7,47.1,East,No
6,7,48,92.9,60.0,East,Yes
7,8,21,53.3,36.0,South,No
8,9,39,71.9,48.9,East,Yes
9,10,25,70.4,42.1,East,Yes


**Explain each issue:** what you found and how you handled it.

I found that there was missing values in income, a all lowercase label in region, and a impossible value in income. I chose to fill in the missing data using the median. I replaced the all lowercase label with the correct label, and replaced the impossible value with NaN and used the median to fill in the missing value. 

## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [57]:
# data types: income = ?, spending = ?
# tool: ?
# hint: two continuous variables moving together
from scipy import stats
print("---------------------------------------Income and Spending-------------------------------")
print("Income and Education")
r, p = stats.pearsonr(df['income'], df['spending'])
print(f'r Correlation Coefficients: {r:.3f}')
print(f'p-value:                    {p:.3f}')

---------------------------------------Income and Spending-------------------------------
Income and Education
r Correlation Coefficients: 0.717
p-value:                    0.000


The strength is strong and the direction is positive. The significance of the p-value is highly significant as it indicates that the probability of finding a relationship by chance is 0.


### Q2. Is a customer's region related to whether they subscribed?

In [63]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

contingency_table = pd.crosstab(df['region'], df['subscribed'])
print(contingency_table)  

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table, correction=False) 
print()

expected_df = pd.DataFrame(expected, index = contingency_table.index, columns = contingency_table.columns)
print("Expected Values")
print(expected_df.round(2))





subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1

Expected Values
subscribed    No   Yes
region                
East        10.5  10.5
South        4.5   4.5
West         5.0   5.0


Customer's region is related to whether they are subscribed


### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [59]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

x = df['age']
y = df['income']

x = sm.add_constant(x)

model = sm.OLS(y, x).fit()

predictions = model.predict(x) # use it later for visualization
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 income   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.640
Method:                 Least Squares   F-statistic:                     70.29
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           3.60e-10
Time:                        21:59:53   Log-Likelihood:                -146.41
No. Observations:                  40   AIC:                             296.8
Df Residuals:                      38   BIC:                             300.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         40.2539      5.719      7.039      0.0

I think we can predict 64% of the variation, but the other 36% is due from random chance or other variables not included. 

### Q4. Which region has the most customers, and what is the average income overall?

In [60]:
# hint: count a categorical column
# take the mean of a numeric column
categorical = df['region'].value_counts()
mean = df['income'].mean()

print(categorical)
print()
print(f"The average income is: {mean}")

region
East     21
West     10
South     9
Name: count, dtype: int64

The average income is: 86.46


The region with the most customers is is East. The average income overall is 86.46


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

What decided which tool I used, was based on what the question was asking for. For example, question 4 was asking for predictions, which is obviously R-squared and p-value 


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

No, it only tells how strong the 2 variables change together, not whether one changes because of the other. 


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [64]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?

alpha = 0.05

critical_value = stats.chi2.ppf(1 - alpha, dof)
print(f"Critical value (alpha = {alpha} and degree of freedom = {dof}) is: {critical_value:.2f}")
print(F"Our Chi-square= {chi2}")

if chi2 > critical_value:
  print(f"Since {chi2:.2f} > {critical_value:.2f}, Reject independence (reject the null hypothesis) -> region and subcribed ARE related")
else:
  print(f"Since {chi2:.2f} <= {critical_value:.2f}, Cannot reject")


Critical value (alpha = 0.05 and degree of freedom = 2) is: 5.99
Our Chi-square= 13.161904761904761
Since 13.16 > 5.99, Reject independence (reject the null hypothesis) -> region and subcribed ARE related


I think I would tell the company that the result should be trusted.

### Push your code to GitHub